# Notebook 03 — Model Architecture

## Building a Foundation Model from Scratch

This notebook defines, implements, and validates the decoder-only Transformer architecture used in the controlled model-scaling experiment.

### Experimental architecture family

We will implement three progressively larger members of the same architectural family while holding the tokenizer, dataset, context length, training objective, and training methodology constant.

| Model | Layers | d_model | Heads | Head Dim | SwiGLU Hidden | Target Scale |
|---|---:|---:|---:|---:|---:|---:|
| A | 4 | 256 | 4 | 64 | 704 | ~7M |
| B | 6 | 384 | 6 | 64 | 1,024 | ~17M |
| C | 8 | 512 | 8 | 64 | 1,360 | ~34M |

### Core architecture

Each model uses:

- learned token embeddings
- causal multi-head self-attention
- Rotary Position Embeddings (RoPE)
- RMSNorm
- SwiGLU feed-forward networks
- pre-normalization residual blocks
- final RMSNorm
- tied token-embedding / output-projection weights

The implementation is written explicitly in PyTorch rather than using a prebuilt Transformer model.

### Fixed model-level controls

- Vocabulary size: 16,384
- Context length: 512 tokens
- Dropout: 0.10
- Attention head dimension: 64
- Linear projection biases: disabled
- Input/output embedding weights: tied


In [1]:
from dataclasses import dataclass

VOCAB_SIZE = 16_384
CONTEXT_LENGTH = 512
DROPOUT = 0.10


@dataclass(frozen=True)
class ModelConfig:
    name: str
    vocab_size: int
    context_length: int
    n_layers: int
    d_model: int
    n_heads: int
    d_ff: int
    dropout: float = DROPOUT

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads


MODEL_CONFIGS = {
    "A": ModelConfig(
        name="Model A",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=4,
        d_model=256,
        n_heads=4,
        d_ff=704,
    ),
    "B": ModelConfig(
        name="Model B",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=6,
        d_model=384,
        n_heads=6,
        d_ff=1024,
    ),
    "C": ModelConfig(
        name="Model C",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=8,
        d_model=512,
        n_heads=8,
        d_ff=1360,
    ),
}

MODEL_CONFIGS


{'A': ModelConfig(name='Model A', vocab_size=16384, context_length=512, n_layers=4, d_model=256, n_heads=4, d_ff=704, dropout=0.1),
 'B': ModelConfig(name='Model B', vocab_size=16384, context_length=512, n_layers=6, d_model=384, n_heads=6, d_ff=1024, dropout=0.1),
 'C': ModelConfig(name='Model C', vocab_size=16384, context_length=512, n_layers=8, d_model=512, n_heads=8, d_ff=1360, dropout=0.1)}

In [2]:
for key, cfg in MODEL_CONFIGS.items():
    assert cfg.d_model % cfg.n_heads == 0
    assert cfg.head_dim == 64
    assert cfg.context_length == CONTEXT_LENGTH
    assert cfg.vocab_size == VOCAB_SIZE

    print(
        f"{cfg.name}: "
        f"{cfg.n_layers} layers, "
        f"d_model={cfg.d_model}, "
        f"{cfg.n_heads} heads × {cfg.head_dim} dims, "
        f"d_ff={cfg.d_ff}"
    )


Model A: 4 layers, d_model=256, 4 heads × 64 dims, d_ff=704
Model B: 6 layers, d_model=384, 6 heads × 64 dims, d_ff=1024
Model C: 8 layers, d_model=512, 8 heads × 64 dims, d_ff=1360


In [3]:
def analytical_parameter_count(cfg: ModelConfig) -> dict:
    embeddings = cfg.vocab_size * cfg.d_model
    attention_per_layer = 4 * cfg.d_model**2
    swiglu_per_layer = 3 * cfg.d_model * cfg.d_ff
    norms_per_layer = 2 * cfg.d_model
    block_per_layer = attention_per_layer + swiglu_per_layer + norms_per_layer
    transformer_blocks = cfg.n_layers * block_per_layer
    final_norm = cfg.d_model
    total = embeddings + transformer_blocks + final_norm
    return {"embeddings": embeddings, "attention_per_layer": attention_per_layer, "swiglu_per_layer": swiglu_per_layer, "norms_per_layer": norms_per_layer, "block_per_layer": block_per_layer, "transformer_blocks": transformer_blocks, "final_norm": final_norm, "total": total}

for key, cfg in MODEL_CONFIGS.items():
    counts = analytical_parameter_count(cfg)
    print(f"{cfg.name}: {counts['total']:,} parameters ({counts['total'] / 1e6:.2f}M)")


Model A: 7,407,872 parameters (7.41M)
Model B: 16,913,280 parameters (16.91M)
Model C: 33,497,600 parameters (33.50M)


## Parameter-count mental model

For each Transformer block:

- Attention contributes `4 * d_model^2` parameters for Q, K, V, and output projections.
- SwiGLU contributes `3 * d_model * d_ff` parameters for gate, up, and down projections.
- Two RMSNorms contribute `2 * d_model` learned scale parameters.

The token embedding matrix contributes `vocab_size * d_model` parameters and is tied to the language-model output projection.

RoPE adds no learned parameters.

For Model A, the embedding matrix alone contains 4,194,304 parameters, which is about 57% of the full 7.41M-parameter model. This is why vocabulary size and weight tying matter materially at small model scales.


## Chunk 2 — RMSNorm

Before implementing attention or the feed-forward network, we implement the normalization used throughout the model.

### Why normalization is needed

As activations move through many residual blocks, their scale can drift. Normalization keeps the numerical scale of those activations controlled, which generally makes optimization more stable.

A classic **LayerNorm** normalizes using both the mean and variance of the hidden features. RMSNorm is simpler: it rescales the vector using its root-mean-square magnitude without subtracting the mean.

Our architecture uses two RMSNorms inside every Transformer block and one final RMSNorm after the last block.


In [4]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype
        x_float = x.float()
        mean_square = x_float.pow(2).mean(dim=-1, keepdim=True)
        x_normalized = x_float * torch.rsqrt(mean_square + self.eps)
        return (x_normalized * self.weight).to(dtype=input_dtype)


In [5]:
torch.manual_seed(42)
cfg = MODEL_CONFIGS["A"]
norm = RMSNorm(cfg.d_model)
x = torch.randn(2, 5, cfg.d_model)
y = norm(x)
assert y.shape == x.shape
parameter_count = sum(p.numel() for p in norm.parameters())
assert parameter_count == cfg.d_model
output_rms = y.float().pow(2).mean(dim=-1).sqrt()
max_deviation_from_one = (output_rms - 1.0).abs().max().item()
assert max_deviation_from_one < 1e-5
print(f"Input shape:              {tuple(x.shape)}")
print(f"Output shape:             {tuple(y.shape)}")
print(f"RMSNorm parameters:       {parameter_count:,}")
print(f"Expected parameters:      {cfg.d_model:,}")
print(f"Max RMS deviation from 1: {max_deviation_from_one:.2e}")


Input shape:              (2, 5, 256)
Output shape:             (2, 5, 256)
RMSNorm parameters:       256
Expected parameters:      256
Max RMS deviation from 1: 5.96e-07


In [6]:
for key, cfg in MODEL_CONFIGS.items():
    test_norm = RMSNorm(cfg.d_model)
    observed = sum(p.numel() for p in test_norm.parameters())
    expected = cfg.d_model
    assert observed == expected
    print(f"{cfg.name}: RMSNorm = {observed:,} learned parameters")


Model A: RMSNorm = 256 learned parameters
Model B: RMSNorm = 384 learned parameters
Model C: RMSNorm = 512 learned parameters


### RMSNorm mental model
> **RMSNorm controls the magnitude of the hidden-state vector without recentering it.**


## Chunk 3 — Rotary Position Embeddings (RoPE)

RoPE injects positional information by rotating attention query and key vectors as a deterministic function of position. RoPE adds no learned parameters.


In [7]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, base: float = 10_000.0):
        super().__init__()
        if head_dim % 2 != 0:
            raise ValueError("RoPE requires an even head dimension.")
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        pair_dims = torch.arange(0, head_dim, 2, dtype=torch.float32)
        inv_freq = base ** (-pair_dims / head_dim)
        positions = torch.arange(max_seq_len, dtype=torch.float32)
        angles = torch.outer(positions, inv_freq)
        self.register_buffer("cos_cached", angles.cos(), persistent=False)
        self.register_buffer("sin_cached", angles.sin(), persistent=False)
    @staticmethod
    def _apply_rotation(x, cos, sin):
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]
        return torch.stack((x_even * cos - x_odd * sin, x_even * sin + x_odd * cos), dim=-1).flatten(-2)
    def forward(self, q, k):
        seq_len = q.size(-2)
        cos = self.cos_cached[:seq_len].to(device=q.device, dtype=q.dtype)[None, None, :, :]
        sin = self.sin_cached[:seq_len].to(device=q.device, dtype=q.dtype)[None, None, :, :]
        return self._apply_rotation(q, cos, sin), self._apply_rotation(k, cos, sin)


In [8]:
torch.manual_seed(42)
cfg = MODEL_CONFIGS["A"]
rope = RotaryEmbedding(cfg.head_dim, cfg.context_length)
q = torch.randn(2, cfg.n_heads, 16, cfg.head_dim)
k = torch.randn(2, cfg.n_heads, 16, cfg.head_dim)
q_rot, k_rot = rope(q, k)
assert q_rot.shape == q.shape and k_rot.shape == k.shape
assert sum(p.numel() for p in rope.parameters()) == 0
zero_position_error = (q_rot[:, :, 0] - q[:, :, 0]).abs().max().item()
assert zero_position_error == 0.0
norm_error = (q_rot.float().norm(dim=-1) - q.float().norm(dim=-1)).abs().max().item()
assert norm_error < 1e-5
print(f"Q shape:                    {tuple(q.shape)}")
print(f"Rotated Q shape:            {tuple(q_rot.shape)}")
print(f"RoPE learned parameters:    0")
print(f"Position-0 identity error:  {zero_position_error:.2e}")
print(f"Max norm-preservation error: {norm_error:.2e}")


Q shape:                    (2, 4, 16, 64)
Rotated Q shape:            (2, 4, 16, 64)
RoPE learned parameters:    0
Position-0 identity error:  0.00e+00
Max norm-preservation error: 9.54e-07


In [9]:
torch.manual_seed(42)
seq_len = 16
q_content = torch.randn(cfg.head_dim)
k_content = torch.randn(cfg.head_dim)
q_same = q_content.view(1,1,1,-1).expand(1,1,seq_len,-1).clone()
k_same = k_content.view(1,1,1,-1).expand(1,1,seq_len,-1).clone()
q_same_rot, k_same_rot = rope(q_same, k_same)
dot_2_5 = torch.dot(q_same_rot[0,0,2], k_same_rot[0,0,5])
dot_7_10 = torch.dot(q_same_rot[0,0,7], k_same_rot[0,0,10])
relative_position_error = (dot_2_5 - dot_7_10).abs().item()
assert relative_position_error < 1e-5
print(f"Dot product at positions 2→5:  {dot_2_5.item():.6f}")
print(f"Dot product at positions 7→10: {dot_7_10.item():.6f}")
print(f"Difference:                     {relative_position_error:.2e}")


Dot product at positions 2→5:  -6.535116
Dot product at positions 7→10: -6.535115
Difference:                     1.43e-06


In [10]:
for key, model_cfg in MODEL_CONFIGS.items():
    assert model_cfg.head_dim == 64
    test_rope = RotaryEmbedding(model_cfg.head_dim, model_cfg.context_length)
    assert sum(p.numel() for p in test_rope.parameters()) == 0
    print(f"{model_cfg.name}: head_dim={model_cfg.head_dim}, context={model_cfg.context_length}, RoPE parameters=0")


Model A: head_dim=64, context=512, RoPE parameters=0
Model B: head_dim=64, context=512, RoPE parameters=0
Model C: head_dim=64, context=512, RoPE parameters=0


### RoPE mental model
> **RoPE encodes position by rotating query and key vectors, so attention can become sensitive to relative token positions without learning a separate positional embedding table.**


## Chunk 4 — Causal Multi-Head Self-Attention

Attention lets each token route information from earlier visible tokens. Queries and keys determine compatibility, values carry the retrieved information, and a causal mask prevents future-token leakage.


In [11]:
import math

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.d_model = cfg.d_model
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.head_dim
        self.max_seq_len = cfg.context_length
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, self.max_seq_len)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask[None,None,:,:], persistent=False)
    def _split_heads(self, x):
        b,t,_=x.shape
        return x.view(b,t,self.n_heads,self.head_dim).transpose(1,2)
    def _merge_heads(self,x):
        b,_,t,_=x.shape
        return x.transpose(1,2).contiguous().view(b,t,self.d_model)
    def forward(self,x,return_attention=False):
        _,t,_=x.shape
        q=self._split_heads(self.q_proj(x)); k=self._split_heads(self.k_proj(x)); v=self._split_heads(self.v_proj(x))
        q,k=self.rope(q,k)
        scores=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.head_dim)
        mask=self.causal_mask[:,:,:t,:t]
        scores=scores.masked_fill(~mask,float('-inf'))
        attention=torch.softmax(scores.float(),dim=-1).to(dtype=v.dtype)
        attention=self.attn_dropout(attention)
        context=torch.matmul(attention,v)
        output=self.resid_dropout(self.out_proj(self._merge_heads(context)))
        return (output,attention) if return_attention else output


In [12]:
torch.manual_seed(42)
cfg=MODEL_CONFIGS['A']
attn=CausalSelfAttention(cfg); attn.eval()
x=torch.randn(2,16,cfg.d_model)
with torch.no_grad(): y,weights=attn(x,return_attention=True)
assert y.shape==x.shape
observed_parameters=sum(p.numel() for p in attn.parameters())
expected_parameters=4*cfg.d_model**2
assert observed_parameters==expected_parameters
print(f"Input shape:                 {tuple(x.shape)}")
print(f"Output shape:                {tuple(y.shape)}")
print(f"Attention-weight shape:      {tuple(weights.shape)}")
print(f"Observed attention params:   {observed_parameters:,}")
print(f"Expected attention params:   {expected_parameters:,}")


Input shape:                 (2, 16, 256)
Output shape:                (2, 16, 256)
Attention-weight shape:      (2, 4, 16, 16)
Observed attention params:   262,144
Expected attention params:   262,144


In [13]:
row_sums=weights.float().sum(dim=-1)
row_sum_error=(row_sums-1.0).abs().max().item()
future_mask=torch.triu(torch.ones(16,16,dtype=torch.bool),diagonal=1)
max_future_attention=weights[...,future_mask].abs().max().item()
assert row_sum_error<1e-6 and max_future_attention==0.0
print(f"Max attention-row sum error: {row_sum_error:.2e}")
print(f"Max future attention weight: {max_future_attention:.2e}")


Max attention-row sum error: 1.79e-07
Max future attention weight: 0.00e+00


In [14]:
torch.manual_seed(42)
test_x=torch.randn(1,8,cfg.d_model); changed_x=test_x.clone(); changed_x[:,7,:]=torch.randn_like(changed_x[:,7,:])*100.0
with torch.no_grad(): original_output=attn(test_x); changed_output=attn(changed_x)
earlier_difference=(original_output[:,:7]-changed_output[:,:7]).abs().max().item()
final_position_difference=(original_output[:,7]-changed_output[:,7]).abs().max().item()
assert earlier_difference<1e-6 and final_position_difference>1e-3
print(f"Max difference at positions 0..6: {earlier_difference:.2e}")
print(f"Difference at changed position 7:  {final_position_difference:.2e}")


Max difference at positions 0..6: 0.00e+00
Difference at changed position 7:  9.08e+01


In [15]:
for key, model_cfg in MODEL_CONFIGS.items():
    test_attn=CausalSelfAttention(model_cfg)
    observed=sum(p.numel() for p in test_attn.parameters()); expected=4*model_cfg.d_model**2
    assert observed==expected
    print(f"{model_cfg.name}: {model_cfg.n_heads} heads × {model_cfg.head_dim} dims, attention parameters={observed:,}")


Model A: 4 heads × 64 dims, attention parameters=262,144
Model B: 6 heads × 64 dims, attention parameters=589,824
Model C: 8 heads × 64 dims, attention parameters=1,048,576


### Attention mental model
> **Attention routes information between tokens. Q and K determine where to look, V determines what is retrieved, and the causal mask prevents information from flowing backward from the future.**


## Chunk 5 — SwiGLU Feed-Forward Network

Attention mixes information across token positions. SwiGLU instead transforms each token's hidden features independently.


In [16]:
import torch.nn.functional as F

class SwiGLU(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.gate_proj=nn.Linear(cfg.d_model,cfg.d_ff,bias=False)
        self.up_proj=nn.Linear(cfg.d_model,cfg.d_ff,bias=False)
        self.down_proj=nn.Linear(cfg.d_ff,cfg.d_model,bias=False)
        self.dropout=nn.Dropout(cfg.dropout)
    def forward(self,x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x))*self.up_proj(x)))


In [17]:
torch.manual_seed(42)
cfg=MODEL_CONFIGS['A']; ffn=SwiGLU(cfg); ffn.eval()
x=torch.randn(2,16,cfg.d_model)
with torch.no_grad(): y=ffn(x)
observed_parameters=sum(p.numel() for p in ffn.parameters()); expected_parameters=3*cfg.d_model*cfg.d_ff
assert y.shape==x.shape and observed_parameters==expected_parameters
print(f"Input shape:               {tuple(x.shape)}")
print(f"Output shape:              {tuple(y.shape)}")
print(f"Model A d_ff:              {cfg.d_ff:,}")
print(f"Observed SwiGLU params:    {observed_parameters:,}")
print(f"Expected SwiGLU params:    {expected_parameters:,}")


Input shape:               (2, 16, 256)
Output shape:              (2, 16, 256)
Model A d_ff:              704
Observed SwiGLU params:    540,672
Expected SwiGLU params:    540,672


In [18]:
torch.manual_seed(42)
test_x=torch.randn(1,8,cfg.d_model); changed_x=test_x.clone(); changed_x[:,4,:]=torch.randn_like(changed_x[:,4,:])*100.0
with torch.no_grad(): original_output=ffn(test_x); changed_output=ffn(changed_x)
idx=[0,1,2,3,5,6,7]
other_token_difference=(original_output[:,idx]-changed_output[:,idx]).abs().max().item()
changed_token_difference=(original_output[:,4]-changed_output[:,4]).abs().max().item()
assert other_token_difference==0.0 and changed_token_difference>1e-3
print(f"Max difference at unchanged tokens: {other_token_difference:.2e}")
print(f"Difference at changed token 4:       {changed_token_difference:.2e}")


Max difference at unchanged tokens: 0.00e+00
Difference at changed token 4:       1.34e+03


In [19]:
probe=torch.randn(1,3,cfg.d_model)
with torch.no_grad():
    gate_values=F.silu(ffn.gate_proj(probe)); up_values=ffn.up_proj(probe); gated_hidden=gate_values*up_values; projected_back=ffn.down_proj(gated_hidden)
print(f"Input:          {tuple(probe.shape)}")
print(f"Gate branch:    {tuple(gate_values.shape)}")
print(f"Up branch:      {tuple(up_values.shape)}")
print(f"Gated hidden:   {tuple(gated_hidden.shape)}")
print(f"Projected back: {tuple(projected_back.shape)}")


Input:          (1, 3, 256)
Gate branch:    (1, 3, 704)
Up branch:      (1, 3, 704)
Gated hidden:   (1, 3, 704)
Projected back: (1, 3, 256)


In [20]:
for key,model_cfg in MODEL_CONFIGS.items():
    test_ffn=SwiGLU(model_cfg); observed=sum(p.numel() for p in test_ffn.parameters()); expected=3*model_cfg.d_model*model_cfg.d_ff
    assert observed==expected
    print(f"{model_cfg.name}: d_model={model_cfg.d_model}, d_ff={model_cfg.d_ff}, SwiGLU parameters={observed:,}")


Model A: d_model=256, d_ff=704, SwiGLU parameters=540,672
Model B: d_model=384, d_ff=1024, SwiGLU parameters=1,179,648
Model C: d_model=512, d_ff=1360, SwiGLU parameters=2,088,960


### SwiGLU mental model
> **Attention lets a token gather information from other tokens. SwiGLU then transforms that gathered information inside the token's own feature vector.**


## Chunk 6 — Complete Pre-Norm Transformer Block

Each block combines RMSNorm, causal attention with RoPE, a residual connection, a second RMSNorm, SwiGLU, and a second residual connection.


In [21]:
class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.attn_norm=RMSNorm(cfg.d_model); self.attn=CausalSelfAttention(cfg)
        self.ffn_norm=RMSNorm(cfg.d_model); self.ffn=SwiGLU(cfg)
    def forward(self,x):
        x=x+self.attn(self.attn_norm(x))
        x=x+self.ffn(self.ffn_norm(x))
        return x


In [22]:
torch.manual_seed(42)
cfg=MODEL_CONFIGS['A']; block=TransformerBlock(cfg); block.eval(); x=torch.randn(2,16,cfg.d_model)
with torch.no_grad(): y=block(x)
observed_parameters=sum(p.numel() for p in block.parameters())
expected_parameters=4*cfg.d_model**2+3*cfg.d_model*cfg.d_ff+2*cfg.d_model
assert y.shape==x.shape and observed_parameters==expected_parameters
print(f"Input shape:               {tuple(x.shape)}")
print(f"Output shape:              {tuple(y.shape)}")
print(f"Observed block params:     {observed_parameters:,}")
print(f"Expected block params:     {expected_parameters:,}")


Input shape:               (2, 16, 256)
Output shape:              (2, 16, 256)
Observed block params:     803,328
Expected block params:     803,328


In [23]:
with torch.no_grad():
    after_attention=x+block.attn(block.attn_norm(x)); manually_reconstructed=after_attention+block.ffn(block.ffn_norm(after_attention))
reconstruction_error=(y-manually_reconstructed).abs().max().item(); assert reconstruction_error==0.0
print(f"Manual block reconstruction error: {reconstruction_error:.2e}")


Manual block reconstruction error: 0.00e+00


In [24]:
torch.manual_seed(42)
test_x=torch.randn(1,8,cfg.d_model); changed_x=test_x.clone(); changed_x[:,7,:]=torch.randn_like(changed_x[:,7,:])*100.0
with torch.no_grad(): original_output=block(test_x); changed_output=block(changed_x)
earlier_difference=(original_output[:,:7]-changed_output[:,:7]).abs().max().item(); final_position_difference=(original_output[:,7]-changed_output[:,7]).abs().max().item()
assert earlier_difference<1e-6 and final_position_difference>1e-3
print(f"Max difference at positions 0..6: {earlier_difference:.2e}")
print(f"Difference at changed position 7:  {final_position_difference:.2e}")


Max difference at positions 0..6: 0.00e+00
Difference at changed position 7:  2.57e+02


In [25]:
for key,model_cfg in MODEL_CONFIGS.items():
    test_block=TransformerBlock(model_cfg); observed=sum(p.numel() for p in test_block.parameters()); expected=4*model_cfg.d_model**2+3*model_cfg.d_model*model_cfg.d_ff+2*model_cfg.d_model
    assert observed==expected
    print(f"{model_cfg.name}: one Transformer block = {observed:,} parameters")


Model A: one Transformer block = 803,328 parameters
Model B: one Transformer block = 1,770,240 parameters
Model C: one Transformer block = 3,138,560 parameters


### Transformer-block mental model
> **A Transformer block alternates between communication and computation. Attention communicates across tokens; SwiGLU computes within each token; residual paths preserve and refine the running representation.**


## Chunk 7 — Complete Decoder-Only Language Model

The full model adds learned token embeddings, a stack of Transformer blocks, a final RMSNorm, and a tied vocabulary projection that produces next-token logits.


In [26]:
class DecoderOnlyLM(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.cfg=cfg
        self.token_embedding=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm=RMSNorm(cfg.d_model)
        self.lm_head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.token_embedding.weight
    def forward(self,input_ids):
        if input_ids.ndim!=2: raise ValueError('input_ids must have shape [batch, sequence].')
        if input_ids.size(1)>self.cfg.context_length: raise ValueError('Sequence length exceeds configured context length.')
        x=self.token_embedding(input_ids)
        for block in self.blocks: x=block(x)
        return self.lm_head(self.final_norm(x))


In [27]:
torch.manual_seed(42)
cfg=MODEL_CONFIGS['A']; model=DecoderOnlyLM(cfg); model.eval(); input_ids=torch.randint(0,cfg.vocab_size,(2,16))
with torch.no_grad(): logits=model(input_ids)
assert logits.shape==(2,16,cfg.vocab_size)
print(f"Input token-ID shape: {tuple(input_ids.shape)}")
print(f"Logit shape:          {tuple(logits.shape)}")
print(f"Vocabulary size:      {cfg.vocab_size:,}")


Input token-ID shape: (2, 16)
Logit shape:          (2, 16, 16384)
Vocabulary size:      16,384


In [28]:
same_parameter_object=model.token_embedding.weight is model.lm_head.weight
same_storage=model.token_embedding.weight.data_ptr()==model.lm_head.weight.data_ptr()
assert same_parameter_object and same_storage
embedding_parameters=model.token_embedding.weight.numel()
print(f"Same Parameter object: {same_parameter_object}")
print(f"Same memory storage:   {same_storage}")
print(f"Shared matrix size:    {embedding_parameters:,} parameters")
print(f"Parameters saved vs untied Model A: {embedding_parameters:,}")


Same Parameter object: True
Same memory storage:   True
Shared matrix size:    4,194,304 parameters
Parameters saved vs untied Model A: 4,194,304


In [29]:
observed_total=sum(p.numel() for p in model.parameters()); expected_total=analytical_parameter_count(cfg)['total']; assert observed_total==expected_total
print(f"Observed Model A parameters: {observed_total:,}")
print(f"Analytical expectation:      {expected_total:,}")
print(f"Difference:                  {observed_total-expected_total:,}")


Observed Model A parameters: 7,407,872
Analytical expectation:      7,407,872
Difference:                  0


In [30]:
torch.manual_seed(42)
test_ids=torch.randint(0,cfg.vocab_size,(1,8)); changed_ids=test_ids.clone(); changed_ids[0,7]=(changed_ids[0,7]+1)%cfg.vocab_size
with torch.no_grad(): original_logits=model(test_ids); changed_logits=model(changed_ids)
earlier_logit_difference=(original_logits[:,:7]-changed_logits[:,:7]).abs().max().item(); final_logit_difference=(original_logits[:,7]-changed_logits[:,7]).abs().max().item()
assert earlier_logit_difference<1e-6 and final_logit_difference>1e-6
print(f"Max logit difference at positions 0..6: {earlier_logit_difference:.2e}")
print(f"Difference at changed position 7:        {final_logit_difference:.2e}")


Max logit difference at positions 0..6: 0.00e+00
Difference at changed position 7:        5.33e+01


In [31]:
for key,model_cfg in MODEL_CONFIGS.items():
    family_model=DecoderOnlyLM(model_cfg); observed=sum(p.numel() for p in family_model.parameters()); expected=analytical_parameter_count(model_cfg)['total']
    assert observed==expected and family_model.token_embedding.weight is family_model.lm_head.weight
    print(f"{model_cfg.name}: {observed:,} parameters ({observed/1e6:.2f}M)")
    del family_model


Model A: 7,407,872 parameters (7.41M)
Model B: 16,913,280 parameters (16.91M)
Model C: 33,497,600 parameters (33.50M)


### Complete-model mental model
> **The language model converts token IDs into contextual hidden representations and converts each contextual representation into scores over the vocabulary for the next token.**


## Chunk 8 — Explicit Weight Initialization

Initialization is explicit rather than inherited from PyTorch defaults. Ordinary embeddings and linear weights use `Normal(0, 0.02)`, RMSNorm scales start at 1, and residual-output projections use depth-aware scaling `0.02 / sqrt(2L)`.


In [32]:
INIT_STD=0.02
MODEL_SEED=42

def initialize_model_weights(model,seed=MODEL_SEED,base_std=INIT_STD):
    torch.manual_seed(seed); cfg=model.cfg
    nn.init.normal_(model.token_embedding.weight,mean=0.0,std=base_std)
    residual_std=base_std/math.sqrt(2*cfg.n_layers)
    for block in model.blocks:
        nn.init.ones_(block.attn_norm.weight); nn.init.ones_(block.ffn_norm.weight)
        nn.init.normal_(block.attn.q_proj.weight,0.0,base_std); nn.init.normal_(block.attn.k_proj.weight,0.0,base_std); nn.init.normal_(block.attn.v_proj.weight,0.0,base_std)
        nn.init.normal_(block.attn.out_proj.weight,0.0,residual_std)
        nn.init.normal_(block.ffn.gate_proj.weight,0.0,base_std); nn.init.normal_(block.ffn.up_proj.weight,0.0,base_std)
        nn.init.normal_(block.ffn.down_proj.weight,0.0,residual_std)
    nn.init.ones_(model.final_norm.weight)


In [33]:
for key,model_cfg in MODEL_CONFIGS.items():
    residual_std=INIT_STD/math.sqrt(2*model_cfg.n_layers)
    print(f"{model_cfg.name}: base std={INIT_STD:.6f}, residual-output std={residual_std:.6f}")


Model A: base std=0.020000, residual-output std=0.007071
Model B: base std=0.020000, residual-output std=0.005774
Model C: base std=0.020000, residual-output std=0.005000


In [34]:
cfg=MODEL_CONFIGS['A']; initialized_model=DecoderOnlyLM(cfg); initialize_model_weights(initialized_model); initialized_model.eval()
base_target=INIT_STD; residual_target=INIT_STD/math.sqrt(2*cfg.n_layers)
embedding_std=initialized_model.token_embedding.weight.std().item(); q_proj_std=initialized_model.blocks[0].attn.q_proj.weight.std().item(); attn_out_std=initialized_model.blocks[0].attn.out_proj.weight.std().item(); ffn_down_std=initialized_model.blocks[0].ffn.down_proj.weight.std().item()
assert abs(embedding_std-base_target)/base_target<0.05; assert abs(q_proj_std-base_target)/base_target<0.05; assert abs(attn_out_std-residual_target)/residual_target<0.05; assert abs(ffn_down_std-residual_target)/residual_target<0.05
print(f"Embedding std:              {embedding_std:.6f}  (target {base_target:.6f})")
print(f"Q-projection std:           {q_proj_std:.6f}  (target {base_target:.6f})")
print(f"Attention output std:       {attn_out_std:.6f}  (target {residual_target:.6f})")
print(f"SwiGLU down-projection std: {ffn_down_std:.6f}  (target {residual_target:.6f})")


Embedding std:              0.020002  (target 0.020000)
Q-projection std:           0.020044  (target 0.020000)
Attention output std:       0.007069  (target 0.007071)
SwiGLU down-projection std: 0.007062  (target 0.007071)


In [35]:
assert initialized_model.token_embedding.weight is initialized_model.lm_head.weight
observed_total=sum(p.numel() for p in initialized_model.parameters()); expected_total=analytical_parameter_count(cfg)['total']; assert observed_total==expected_total
print('Weight tying preserved:    True')
print(f"Parameter count preserved: {observed_total:,}")


Weight tying preserved:    True
Parameter count preserved: 7,407,872


In [36]:
model_1=DecoderOnlyLM(cfg); model_2=DecoderOnlyLM(cfg); initialize_model_weights(model_1,42); initialize_model_weights(model_2,42)
assert torch.equal(model_1.token_embedding.weight,model_2.token_embedding.weight)
assert torch.equal(model_1.blocks[0].attn.q_proj.weight,model_2.blocks[0].attn.q_proj.weight)
assert torch.equal(model_1.blocks[-1].ffn.down_proj.weight,model_2.blocks[-1].ffn.down_proj.weight)
print('Same seed produces identical checked weights: True')
del model_1,model_2


Same seed produces identical checked weights: True


In [37]:
torch.manual_seed(42); probe_ids=torch.randint(0,cfg.vocab_size,(2,16))
with torch.no_grad(): probe_logits=initialized_model(probe_ids)
assert probe_logits.shape==(2,16,cfg.vocab_size) and torch.isfinite(probe_logits).all()
print(f"Logit shape:          {tuple(probe_logits.shape)}")
print(f"All logits finite:    {torch.isfinite(probe_logits).all().item()}")
print(f"Initial logit std:    {probe_logits.std().item():.6f}")


Logit shape:          (2, 16, 16384)
All logits finite:    True
Initial logit std:    0.318682


### Initialization mental model
> **Initialization chooses the model's starting geometry before learning begins. Ordinary weights start at std 0.02; residual-output weights start smaller by `1 / sqrt(2L)`.**


## Chunk 9 — Final Architecture Audit

No new architecture is introduced here. The audit verifies that implementation and locked specification agree before Notebook 03 is closed.


In [38]:
EXPECTED_ARCHITECTURE={
 'A':{'n_layers':4,'d_model':256,'n_heads':4,'head_dim':64,'d_ff':704,'parameters':7_407_872},
 'B':{'n_layers':6,'d_model':384,'n_heads':6,'head_dim':64,'d_ff':1024,'parameters':16_913_280},
 'C':{'n_layers':8,'d_model':512,'n_heads':8,'head_dim':64,'d_ff':1360,'parameters':33_497_600},
}
print('Locked architecture specification loaded.')


Locked architecture specification loaded.


In [39]:
def audit_model_structure(key,cfg):
    expected=EXPECTED_ARCHITECTURE[key]; model=DecoderOnlyLM(cfg); initialize_model_weights(model,seed=MODEL_SEED); model.eval()
    assert cfg.vocab_size==16_384 and cfg.context_length==512 and cfg.dropout==0.10
    assert cfg.n_layers==expected['n_layers'] and cfg.d_model==expected['d_model'] and cfg.n_heads==expected['n_heads'] and cfg.head_dim==expected['head_dim']==64 and cfg.d_ff==expected['d_ff']
    assert len(model.blocks)==cfg.n_layers
    embedding_modules=[m for m in model.modules() if isinstance(m,nn.Embedding)]; assert len(embedding_modules)==1 and embedding_modules[0] is model.token_embedding
    rmsnorm_modules=[m for m in model.modules() if isinstance(m,RMSNorm)]; assert len(rmsnorm_modules)==2*cfg.n_layers+1
    for block in model.blocks:
        assert isinstance(block.attn.rope,RotaryEmbedding) and sum(p.numel() for p in block.attn.rope.parameters())==0
        assert block.attn.rope.cos_cached.shape==(cfg.context_length,cfg.head_dim//2)
    linear_modules=[m for m in model.modules() if isinstance(m,nn.Linear)]; assert linear_modules and all(m.bias is None for m in linear_modules)
    dropout_modules=[m for m in model.modules() if isinstance(m,nn.Dropout)]; assert dropout_modules and all(m.p==cfg.dropout==0.10 for m in dropout_modules)
    assert model.token_embedding.weight is model.lm_head.weight and model.token_embedding.weight.data_ptr()==model.lm_head.weight.data_ptr()
    observed=sum(p.numel() for p in model.parameters()); analytical=analytical_parameter_count(cfg)['total']; assert observed==analytical==expected['parameters']
    residual_target=INIT_STD/math.sqrt(2*cfg.n_layers)
    embedding_std=model.token_embedding.weight.std().item(); q_std=model.blocks[0].attn.q_proj.weight.std().item(); attn_out_std=model.blocks[0].attn.out_proj.weight.std().item(); ffn_down_std=model.blocks[0].ffn.down_proj.weight.std().item()
    assert abs(embedding_std-INIT_STD)/INIT_STD<0.05 and abs(q_std-INIT_STD)/INIT_STD<0.05
    assert abs(attn_out_std-residual_target)/residual_target<0.05 and abs(ffn_down_std-residual_target)/residual_target<0.05
    torch.manual_seed(42); ids=torch.randint(0,cfg.vocab_size,(1,4))
    with torch.no_grad(): logits=model(ids)
    assert logits.shape==(1,4,cfg.vocab_size) and torch.isfinite(logits).all()
    return {'model':cfg.name,'layers':cfg.n_layers,'d_model':cfg.d_model,'heads':cfg.n_heads,'d_ff':cfg.d_ff,'parameters':observed,'residual_std':residual_target}

audit_results=[]
for key,cfg in MODEL_CONFIGS.items():
    result=audit_model_structure(key,cfg); audit_results.append(result)
    print(f"PASS — {result['model']}: {result['layers']} layers, d_model={result['d_model']}, {result['heads']} heads, d_ff={result['d_ff']}, {result['parameters']:,} parameters")


PASS — Model A: 4 layers, d_model=256, 4 heads, d_ff=704, 7,407,872 parameters
PASS — Model B: 6 layers, d_model=384, 6 heads, d_ff=1024, 16,913,280 parameters
PASS — Model C: 8 layers, d_model=512, 8 heads, d_ff=1360, 33,497,600 parameters


In [40]:
cfg=MODEL_CONFIGS['A']; context_test_model=DecoderOnlyLM(cfg); initialize_model_weights(context_test_model)
too_long=torch.zeros((1,cfg.context_length+1),dtype=torch.long); enforced=False
try: context_test_model(too_long)
except ValueError as exc: enforced='context length' in str(exc).lower()
assert enforced
print('PASS — context length > 512 is rejected before model computation.')
del context_test_model


PASS — context length > 512 is rejected before model computation.


### Backward-pass sanity check

A tiny synthetic next-token objective verifies that gradients flow backward through every major learned component. This is structural validation, not training.


In [41]:
torch.manual_seed(42)
cfg=MODEL_CONFIGS['A']; gradient_model=DecoderOnlyLM(cfg); initialize_model_weights(gradient_model,seed=42); gradient_model.train()
token_batch=torch.randint(0,cfg.vocab_size,(2,9)); inputs=token_batch[:,:-1]; targets=token_batch[:,1:]
logits=gradient_model(inputs); loss=F.cross_entropy(logits.reshape(-1,cfg.vocab_size),targets.reshape(-1)); assert torch.isfinite(loss)
gradient_model.zero_grad(set_to_none=True); loss.backward()
gradient_checks={'tied_embedding_lm_head':gradient_model.token_embedding.weight.grad,'attention_q':gradient_model.blocks[0].attn.q_proj.weight.grad,'attention_out':gradient_model.blocks[0].attn.out_proj.weight.grad,'swiglu_gate':gradient_model.blocks[0].ffn.gate_proj.weight.grad,'swiglu_down':gradient_model.blocks[0].ffn.down_proj.weight.grad,'final_norm':gradient_model.final_norm.weight.grad}
for name,grad in gradient_checks.items():
    assert grad is not None and torch.isfinite(grad).all() and grad.abs().sum().item()>0.0
print(f"Synthetic next-token loss: {loss.item():.6f}")
for name,grad in gradient_checks.items(): print(f"PASS — {name}: grad norm={grad.float().norm().item():.6f}")
del gradient_model


Synthetic next-token loss: 9.693123
PASS — tied_embedding_lm_head: grad norm=6.179075
PASS — attention_q: grad norm=0.162414
PASS — attention_out: grad norm=11.208204
PASS — swiglu_gate: grad norm=1.160382
PASS — swiglu_down: grad norm=3.583875
PASS — final_norm: grad norm=0.077154


In [42]:
print('='*66)
print('NOTEBOOK 03 — FINAL ARCHITECTURE AUDIT: PASS')
print('='*66)
print('Vocabulary:              16,384')
print('Context length:          512')
print('Architecture:            decoder-only, pre-norm')
print('Attention:               causal MHA + RoPE')
print('Normalization:           RMSNorm')
print('Feed-forward:            SwiGLU')
print('Linear biases:           disabled')
print('Dropout:                 0.10')
print('Embedding / LM head:     tied')
print('Initialization:          N(0, 0.02) + residual depth scaling')
print('Seed:                    42')
print()
for result in audit_results: print(f"{result['model']}: {result['parameters']:,} parameters ({result['parameters']/1e6:.2f}M)")


NOTEBOOK 03 — FINAL ARCHITECTURE AUDIT: PASS
Vocabulary:              16,384
Context length:          512
Architecture:            decoder-only, pre-norm
Attention:               causal MHA + RoPE
Normalization:           RMSNorm
Feed-forward:            SwiGLU
Linear biases:           disabled
Dropout:                 0.10
Embedding / LM head:     tied
Initialization:          N(0, 0.02) + residual depth scaling
Seed:                    42

Model A: 7,407,872 parameters (7.41M)
Model B: 16,913,280 parameters (16.91M)
Model C: 33,497,600 parameters (33.50M)


## Notebook 03 conclusion

Notebook 03 now contains the complete decoder-only Transformer architecture used in the controlled scaling experiment and validates that implementation matches specification.

### Final architecture family

| Model | Layers | d_model | Heads | Head Dim | d_ff | Parameters |
|---|---:|---:|---:|---:|---:|---:|
| A | 4 | 256 | 4 | 64 | 704 | 7,407,872 |
| B | 6 | 384 | 6 | 64 | 1,024 | 16,913,280 |
| C | 8 | 512 | 8 | 64 | 1,360 | 33,497,600 |

### Architecture locked for training

- learned token embeddings
- causal standard multi-head self-attention
- RoPE applied to Q and K
- RMSNorm
- SwiGLU
- pre-norm residual blocks
- final RMSNorm
- tied embedding / output weights
- no linear biases
- dropout = 0.10
- explicit depth-aware initialization

The next notebook should move from **what the model is** to **how the model is trained**: packing the 20M-token corpus into causal input/target sequences, constructing batches, defining the loss and optimizer, and building the explicit PyTorch training loop.
